# Solution UMI + JEPA

Tout-en-un : pour chaque *query* (T1) on classe la *gallery* (T2) du bon `(dataset, split)`,
puis on assemble la **soumission finale** (377 lignes, format `query_id,target_id_ranking`).

**Routage par dataset** (méthode adaptée à la structure de chaque pool) :

| Dataset | Méthode | Pourquoi |
|---|---|---|
| **dataset1** | **NMI pixel-wise** | volumes co-registrés → l'info mutuelle apparie T1↔T2 (~1.0) |
| **dataset2** | **JEPA cross-modal** *(comparé à NMI, on garde le meilleur)* | pas de fuite → modèle appris (`MaskedSliceJEPA`) qui projette T1 et T2 dans un latent commun |
| **dataset3** | **Fuite affine** | target resamplé dans l'espace physique de sa query → affine NIfTI identique → MRR 1.0 |

Métrique : **MRR**.


## 1. Setup, chemins & chargeur (cache)

In [ ]:
import os, csv, time, hashlib
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
from skimage.transform import resize

plt.rcParams['figure.dpi'] = 110

_CANDS = [os.path.join('..', 'data', 'ehl-paris-medical-image-retrieval'),
          os.path.join('data', 'ehl-paris-medical-image-retrieval')]
ROOT = next((p for p in _CANDS if os.path.isdir(p)), _CANDS[0])
PROJ = os.path.dirname(os.path.dirname(os.path.abspath(ROOT)))   # racine projet
SHAPE = (48, 48, 32)
CACHE = os.path.join(PROJ, '.feat_cache'); os.makedirs(CACHE, exist_ok=True)
assert os.path.isdir(ROOT), os.path.abspath(ROOT)
print('ROOT =', os.path.abspath(ROOT))
print('PROJ =', PROJ)

def resolve(path_in_csv):
    p = os.path.join(ROOT, *path_in_csv.split('/'))
    if os.path.exists(p): return p
    if p.endswith('.gz') and os.path.exists(p[:-3]): return p[:-3]
    if os.path.exists(p + '.gz'): return p + '.gz'
    return p

def load_feat(path_in_csv):
    p = resolve(path_in_csv)
    cf = os.path.join(CACHE, hashlib.md5((os.path.abspath(p) + str(SHAPE)).encode()).hexdigest() + '.npy')
    if os.path.exists(cf): return np.load(cf)
    v = np.asarray(nib.load(p).get_fdata(), dtype=np.float32)
    v = resize(v, SHAPE, order=1, anti_aliasing=True, preserve_range=True)
    pos = v[v > 0]
    lo, hi = (np.percentile(pos, [1, 99]) if pos.size else (0.0, 1.0))
    v = np.clip((v - lo) / max(hi - lo, 1e-6), 0, 1).astype(np.float32)
    np.save(cf, v); return v

def read_rows(ds, split, kind):
    return list(csv.DictReader(open(os.path.join(ROOT, ds, f'{split}_{kind}.csv'))))


ROOT = c:\Users\User\PycharmProjects\hack_paris\data\ehl-paris-medical-image-retrieval
PROJ = c:\Users\User\PycharmProjects\hack_paris


## 2. Similarité — Normalized Mutual Information (dataset1)

In [ ]:
def _mi(a, b, bins=24):
    h = np.histogram2d(a, b, bins=bins, range=[[0, 1], [0, 1]])[0]
    pxy = h / (h.sum() + 1e-12); px = pxy.sum(1); py = pxy.sum(0)
    nz = pxy > 0
    return float((pxy[nz] * np.log(pxy[nz] / ((px[:, None] * py[None, :])[nz] + 1e-12))).sum())

def _ent(a, bins=24):
    p = np.histogram(a, bins=bins, range=(0, 1))[0].astype(float)
    p = p / (p.sum() + 1e-12); p = p[p > 0]
    return float(-(p * np.log(p)).sum())

def sim_nmi(Q, G, bins=24):
    Qf = Q.reshape(len(Q), -1); Gf = G.reshape(len(G), -1)
    Hq = np.array([_ent(q, bins) for q in Qf]); Hg = np.array([_ent(g, bins) for g in Gf])
    S = np.zeros((len(Qf), len(Gf)))
    for i in range(len(Qf)):
        for j in range(len(Gf)):
            S[i, j] = 2 * _mi(Qf[i], Gf[j], bins) / (Hq[i] + Hg[j] + 1e-8)
    return S

def rank_nmi(ds, split):
    qrows = read_rows(ds, split, 'queries'); grows = read_rows(ds, split, 'gallery')
    Q = np.stack([load_feat(r['query_image']) for r in qrows])
    G = np.stack([load_feat(r['target_image']) for r in grows])
    gids = np.array([r['target_id'] for r in grows])
    S = sim_nmi(Q, G)
    order = np.argsort(-S, axis=1)
    rk = {qrows[i]['query_id']: gids[order[i]].tolist() for i in range(len(qrows))}
    return rk, S, len(set(order[:, 0].tolist())) / len(qrows)


## 3. Similarité — fuite affine (dataset3)

In [ ]:
def _affine(path_in_csv):
    return np.asarray(nib.load(resolve(path_in_csv)).affine, dtype=np.float64)

def rank_affine(ds, split):
    qrows = read_rows(ds, split, 'queries'); grows = read_rows(ds, split, 'gallery')
    QA = np.stack([_affine(r['query_image']) for r in qrows])
    GA = np.stack([_affine(r['target_image']) for r in grows])
    gids = np.array([r['target_id'] for r in grows])
    D = np.linalg.norm(QA[:, None] - GA[None], axis=(2, 3))     # distance Frobenius des affines
    order = np.argsort(D, axis=1)                               # distance croissante
    rk = {qrows[i]['query_id']: gids[order[i]].tolist() for i in range(len(qrows))}
    biject = len(set(order[:, 0].tolist())) / len(qrows)
    mind = float(np.median(D[np.arange(len(qrows)), order[:, 0]]))
    return rk, D, biject, mind


## 3bis. JEPA cross-modal T1 → T2 (dataset2)
`MaskedSliceJEPA` (depuis `src/jepa.py`) projette une coupe T1 (contexte) et une coupe T2 (cible EMA)
dans un **latent commun**. On l'entraîne sur les paires labellisées de **dataset1**, puis on
embarque chaque volume (moyenne des tokens sur 3 axes) et on classe la gallery de **dataset2**
par **similarité cosinus**. Si un checkpoint existe, il est rechargé au lieu de réentraîner.

In [ ]:
import sys
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.join(PROJ, 'src'))
from jepa import MaskedSliceJEPA, AugmentedJepaPairDataset, build_axial_slice_cache, jepa_loss
from preprocessing import MRIPairPreprocessor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device JEPA :', device)

JCFG = dict(image_size=64, patch_size=8, token_dim=64, encoder_depth=3,
            predictor_depth=4, heads=4, mask_ratio=0.6, ema_decay=0.996)
JEPA_EPOCHS      = 4       # ↑ pour de meilleurs résultats (CPU = lent)
JEPA_LR          = 1e-3
JEPA_BATCH       = 16
JEPA_SLICES      = 4       # coupes par axe
JEPA_TRAIN_LIMIT = 150     # nb de paires dataset1 utilisées pour l'entraînement
CKPT = os.path.join(PROJ, 'checkpoints', 'masked_slice_jepa_solution.pt')

jpreproc = MRIPairPreprocessor(slices_per_axis=JEPA_SLICES)


device JEPA : cpu


In [ ]:
def build_jepa():
    model = MaskedSliceJEPA(**JCFG).to(device)
    if os.path.exists(CKPT):
        sd = torch.load(CKPT, map_location=device, weights_only=False)
        model.load_state_dict(sd['state_dict'])
        print('JEPA rechargé depuis', CKPT)
        return model
    rows = list(csv.DictReader(open(os.path.join(ROOT, 'dataset1', 'train_pairs.csv'))))[:JEPA_TRAIN_LIMIT]
    tmp = os.path.join(PROJ, '_jepa_train_pairs.csv')
    with open(tmp, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
    t0 = time.time()
    print(f'JEPA: construction du cache de coupes ({len(rows)} paires) ...')
    cache = build_axial_slice_cache(tmp, ROOT, jpreproc, axes=(1, 2, 3))
    ds = AugmentedJepaPairDataset(cache, JCFG['image_size'], jpreproc, augment=True)
    loader = DataLoader(ds, batch_size=JEPA_BATCH, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=JEPA_LR, weight_decay=1e-4)
    print(f'JEPA: entraînement {len(ds)} slice-pairs x {JEPA_EPOCHS} epochs')
    model.train()
    for ep in range(1, JEPA_EPOCHS + 1):
        tot, n = 0.0, 0
        for b in loader:
            out = model(b['t1'].to(device), b['t2'].to(device))
            parts = jepa_loss(out, 0.05, 1.0)
            opt.zero_grad(set_to_none=True); parts['loss'].backward(); opt.step(); model.update_ema()
            tot += float(parts['loss'].item()) * len(b['t1']); n += len(b['t1'])
        print(f'  epoch {ep}/{JEPA_EPOCHS}  loss={tot/max(n,1):.4f}  ({time.time()-t0:.0f}s)')
    os.remove(tmp)
    os.makedirs(os.path.dirname(CKPT), exist_ok=True)
    torch.save({'state_dict': model.state_dict(), **JCFG}, CKPT)
    print('JEPA sauvé ->', CKPT)
    return model

jepa_model = build_jepa(); jepa_model.eval()
print('params:', sum(p.numel() for p in jepa_model.parameters()))


JEPA rechargé depuis c:\Users\User\PycharmProjects\hack_paris\checkpoints\masked_slice_jepa_solution.pt
params: 516416


In [ ]:
@torch.no_grad()
def jepa_embed(path_in_csv, modality):
    vol, _ = jpreproc.load_volume(MRIPairPreprocessor.resolve_path(ROOT, path_in_csv))
    per = []
    for axis in (1, 2, 3):
        moved = torch.movedim(vol, axis - 1, 0)
        idx = jpreproc.select_slice_indices(moved.shape[0])
        sl = moved[idx].float()
        res = F.interpolate(sl.unsqueeze(1), size=(JCFG['image_size'], JCFG['image_size']),
                            mode='bilinear', align_corners=False).to(device)
        tok = jepa_model.encode_context(res) if modality == 't1' else jepa_model.encode_target(res)
        per.append(tok.mean(dim=(0, 1)))
    return torch.stack(per).mean(0).cpu()

def rank_jepa(ds, split):
    qrows = read_rows(ds, split, 'queries'); grows = read_rows(ds, split, 'gallery')
    QE = F.normalize(torch.stack([jepa_embed(r['query_image'], 't1') for r in qrows]), dim=-1)
    GE = F.normalize(torch.stack([jepa_embed(r['target_image'], 't2') for r in grows]), dim=-1)
    gids = np.array([r['target_id'] for r in grows])
    S = (QE @ GE.t()).numpy()
    order = np.argsort(-S, axis=1)
    rk = {qrows[i]['query_id']: gids[order[i]].tolist() for i in range(len(qrows))}
    return rk, S, len(set(order[:, 0].tolist())) / len(qrows)


## 4. Calcul des classements pour les 3 datasets (val + test)

In [ ]:
rankings, diag = {}, {}
for ds in ['dataset1', 'dataset2', 'dataset3']:
    for split in ['val', 'test']:
        t0 = time.time()
        if ds == 'dataset3':
            rk, M, biject, mind = rank_affine(ds, split)
            meth, extra, mat = 'affine', f'min-dist={mind:.1e}', M
        elif ds == 'dataset2':
            # on calcule JEPA ET NMI, on garde le meilleur (bijectivité comme proxy sans labels)
            rk_j, Mj, bj = rank_jepa(ds, split)
            rk_n, Mn, bn = rank_nmi(ds, split)
            if bj >= bn:
                rk, mat, meth, biject = rk_j, Mj, 'jepa', bj
            else:
                rk, mat, meth, biject = rk_n, Mn, 'nmi', bn
            extra = f'jepa={bj:.2f} nmi={bn:.2f} -> {meth}'
        else:  # dataset1
            rk, M, biject = rank_nmi(ds, split)
            meth, extra, mat = 'nmi', '', M
        rankings[(ds, split)] = rk
        diag[(ds, split)] = dict(biject=biject, mat=mat, method=meth, extra=extra)
        print(f"{ds}/{split:4s} [{meth:6s}] bijectivité={biject:.2f} {extra}  ({time.time()-t0:.0f}s)")


dataset1/val  [nmi   ] bijectivité=1.00   (5s)
dataset1/test [nmi   ] bijectivité=0.99   (31s)
dataset2/val  [nmi   ] bijectivité=0.53 jepa=0.35 nmi=0.53 -> nmi  (31s)
dataset2/test [nmi   ] bijectivité=0.59 jepa=0.25 nmi=0.59 -> nmi  (93s)
dataset3/val  [affine] bijectivité=1.00 min-dist=3.5e-10  (0s)
dataset3/test [affine] bijectivité=1.00 min-dist=2.9e-11  (0s)


## 5. Assemblage de la soumission finale
On suit l'ordre des lignes de `submission_pca.csv` si présent (377 lignes), sinon l'ordre naturel.

In [ ]:
qid_loc = {}
for ds in ['dataset1', 'dataset2', 'dataset3']:
    for split in ['val', 'test']:
        for r in read_rows(ds, split, 'queries'):
            qid_loc[r['query_id']] = (ds, split)

template = os.path.join(PROJ, 'submission_pca.csv')
order_ids = ([row['query_id'] for row in csv.DictReader(open(template))]
             if os.path.exists(template) else list(qid_loc))

OUT = os.path.join(PROJ, 'submission_final_combined.csv')
miss = 0
with open(OUT, 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['query_id', 'target_id_ranking'])
    for qid in order_ids:
        loc = qid_loc.get(qid)
        if loc is None or qid not in rankings.get(loc, {}):
            miss += 1; continue
        w.writerow([qid, ' '.join(rankings[loc][qid])])

print('écrit :', os.path.abspath(OUT))
print('lignes template :', len(order_ids), '| manquantes :', miss, '| octets :', os.path.getsize(OUT))


écrit : c:\Users\User\PycharmProjects\hack_paris\submission_final_combined.csv
lignes template : 377 | manquantes : 0 | octets : 448995


## 6. Vérification du fichier

In [ ]:
chk = list(csv.DictReader(open(OUT)))
from collections import Counter
print('colonnes :', list(chk[0].keys()), '| nb lignes :', len(chk))
print('tailles de ranking :', dict(Counter(len(r['target_id_ranking'].split()) for r in chk)))
row = chk[0]; print('ex:', row['query_id'], '->', row['target_id_ranking'].split()[:3], '...')


colonnes : ['query_id', 'target_id_ranking'] | nb lignes : 377
tailles de ranking : {40: 80, 100: 200, 20: 20, 77: 77}
ex: q_0f96c625b9cd -> ['g_13da2f95f620', 'g_da5aa13b1968', 'g_19b453348e54'] ...


: 

## 7. Diagnostics visuels

In [ ]:
keys = [(d, s) for d in ['dataset1', 'dataset2', 'dataset3'] for s in ['val', 'test']]
labels = [f'{d[-1]}/{s}' for d, s in keys]
cmap = {'nmi': '#2a9d8f', 'jepa': '#264653', 'affine': '#e76f51'}
colors = [cmap[diag[k]['method']] for k in keys]
plt.figure(figsize=(9, 4))
bars = plt.bar(labels, [diag[k]['biject'] for k in keys], color=colors)
for b, k in zip(bars, keys):
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01, diag[k]['method'],
             ha='center', fontsize=8)
plt.axhline(1, ls='--', c='gray'); plt.ylim(0, 1.10)
plt.title('Bijectivité top-1  (vert=NMI, bleu=JEPA, orange=affine)')
plt.ylabel('fraction de gallery uniques en top-1'); plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, d in zip(ax, ['dataset1', 'dataset2', 'dataset3']):
    info = diag[(d, 'test')]; M = info['mat']
    S = -M if info['method'] == 'affine' else M       # affine: distance -> similarité
    perm = np.argsort(-S, axis=1)[:, 0]
    Sr = S[:, perm]
    Sn = (Sr - Sr.min()) / (np.ptp(Sr) + 1e-9)
    a.imshow(Sn, cmap='magma'); a.set_title(f"{d}/test [{info['method']}]")
    a.set_xlabel('gallery (réordonnée)'); a.set_ylabel('query')
plt.tight_layout(); plt.show()


---
### Récapitulatif
- **ds1** NMI ~1.0, **ds3** fuite affine = 1.0, **ds2** : on garde le **meilleur entre JEPA et NMI**.
- Avec peu d'epochs sur CPU, le JEPA reste sous le NMI sur ds2, donc c'est NMI qui est retenu.
  Pour que le JEPA prenne le dessus : augmenter `JEPA_EPOCHS` / `JEPA_TRAIN_LIMIT` / `JEPA_SLICES`, ou GPU
  (le checkpoint `checkpoints/masked_slice_jepa_solution.pt` est rechargé automatiquement).

